# 06 Quantise the predictor to 8 bit

Same family of methods as the FPFLI work (github.com/forhadsde/fpfli-32bit-to-4bit-weights-near-full-precision):

1. **Post training quantisation (PTQ):** weights per output channel, symmetric INT8 (levels -127 to 127); activations per tensor INT8 with ranges calibrated on 512 training samples. BatchNorm is folded into the convolution first.
2. **If PTQ loses more than 10 percent accuracy:** quantisation aware training (straight through estimator, `w + (q - w).detach()`) with distillation from the 32 bit model, 10 epochs.

This is fake quantisation in PyTorch: it measures the accuracy effect, not speed on hardware.

In [1]:
%run ./00_common.ipynb
import copy

C:\Users\Rey\anaconda3\envs\kolb\Lib\site-packages\nbformat\validator.py:434: MissingIDFieldWarning: Cell is missing an id field, this will become a hard error in future nbformat versions. You may want to use `normalize()` on your notebooks before validations (available since nbformat 5.1.4). Previous versions of nbformat are fixing this issue transparently, and will stop doing so in the future.
  _validate(nbdict, ref, version, version_minor, relax_add_props)


PROJECT  : C:\Users\Rey\Documents\Kolb_Radar_Project
FAKE_MODE: False
[mitsuba] Warning: Couldn't import the ipywidgets package. Installing this package is required for the system to properly log messages and print in Jupyter notebooks!


torch device: cuda
mitsuba variant: cuda_ad_rgb


Switched off Tx:  2
Switched off Rx:  10
torch 2.5.1+cu121 | device cuda
common ready


In [2]:
if FAKE_MODE and not list(DATA.glob('shard_*.npz')):
    P = sample_params(2000, np.random.default_rng(0)); X = np.stack([fake_features(p) for p in P])
else:
    X, P = load_dataset()
tr, va, te = split_indices(len(X))
fp32 = OneShotNet().to(DEVICE); fp32.load_state_dict(torch.load(MODELS / 'oneshot_fp32.pt', map_location=DEVICE)); fp32.eval()
ref = evaluate(fp32, X[te].astype(np.float32), P[te]); print('FP32 test MAE mm', ref)

FP32 test MAE mm {'x': 2.7962286472320557, 'y': 10.188100814819336, 'z': 5.112585067749023, 'r': 2.5817832946777344}


In [3]:
def fold_bn(model):
    m = copy.deepcopy(model).eval(); feats = list(m.features)
    out = []
    i = 0
    while i < len(feats):
        if isinstance(feats[i], nn.Conv2d) and i + 1 < len(feats) and isinstance(feats[i+1], nn.BatchNorm2d):
            conv, bn = feats[i], feats[i+1]
            w = conv.weight.data; b = conv.bias.data if conv.bias is not None else torch.zeros(w.shape[0], device=w.device)
            s = bn.weight.data / torch.sqrt(bn.running_var + bn.eps)
            new = nn.Conv2d(conv.in_channels, conv.out_channels, conv.kernel_size, padding=conv.padding).to(w.device)
            new.weight.data = w * s[:, None, None, None]; new.bias.data = (b - bn.running_mean) * s + bn.bias.data
            out.append(new); i += 2
        else:
            out.append(feats[i]); i += 1
    m.features = nn.Sequential(*out)
    return m

def q_weight(w, bits=8):
    qmax = 2 ** (bits - 1) - 1
    scale = w.detach().abs().flatten(1).max(1)[0].clamp(min=1e-12) / qmax
    shape = (-1,) + (1,) * (w.dim() - 1)
    q = torch.clamp(torch.round(w / scale.view(shape)), -qmax, qmax) * scale.view(shape)
    return w + (q - w).detach()          # straight through estimator

class QAct(nn.Module):
    """Per tensor INT8 fake quantisation of activations with a calibrated range."""
    def __init__(self, bits=8):
        super().__init__(); self.bits = bits; self.register_buffer('amax', torch.tensor(0.0)); self.calib = True
    def forward(self, x):
        if self.calib:
            self.amax = torch.maximum(self.amax, x.detach().abs().max()); return x
        qmax = 2 ** (self.bits - 1) - 1; s = self.amax.clamp(min=1e-12) / qmax
        q = torch.clamp(torch.round(x / s), -qmax, qmax) * s
        return x + (q - x).detach()

class QModel(nn.Module):
    def __init__(self, folded, wbits=8, abits=8):
        super().__init__(); self.m = folded; self.bits = wbits
        self.qin = QAct(abits); self.qacts = nn.ModuleList()
        for mod in list(self.m.features) + list(self.m.head):
            if isinstance(mod, (nn.Conv2d, nn.Linear)): self.qacts.append(QAct(abits))
    def forward(self, x):
        x = self.qin(x); k = 0
        for mod in list(self.m.features) + list(self.m.head):
            if isinstance(mod, nn.Conv2d):
                x = nn.functional.conv2d(x, q_weight(mod.weight, self.bits), mod.bias, padding=mod.padding); x = self.qacts[k](x); k += 1
            elif isinstance(mod, nn.Linear):
                x = nn.functional.linear(x, q_weight(mod.weight, self.bits), mod.bias); x = self.qacts[k](x); k += 1
            else:
                x = mod(x)
        return x
    def set_calib(self, on):
        for m in self.modules():
            if isinstance(m, QAct): m.calib = on

def build_ptq(model, wbits=8, n_cal=512):
    q = QModel(fold_bn(model), wbits, 8).to(DEVICE).eval(); q.set_calib(True)
    with torch.no_grad():
        xc = torch.from_numpy(X[tr[:n_cal]].astype(np.float32)).to(DEVICE)
        for i in range(0, len(xc), 128): q(xc[i:i+128])
    q.set_calib(False); return q

folded_check = evaluate(fold_bn(fp32), X[te].astype(np.float32), P[te]); print('after BN folding (should equal FP32):', folded_check)
ptq8 = build_ptq(fp32, 8); r_ptq8 = evaluate(ptq8, X[te].astype(np.float32), P[te]); print('PTQ INT8 test MAE mm', r_ptq8)

after BN folding (should equal FP32): {'x': 2.7950937747955322, 'y': 10.188072204589844, 'z': 5.112438678741455, 'r': 2.5818099975585938}
PTQ INT8 test MAE mm {'x': 2.8913683891296387, 'y': 10.187155723571777, 'z': 5.141733169555664, 'r': 2.584380626678467}


In [4]:
def rel_drop(r):
    return float(np.mean([r[k] for k in PARAM_NAMES]) / np.mean([ref[k] for k in PARAM_NAMES]) - 1)

final_name, final_q, final_res = 'PTQ INT8', ptq8, r_ptq8
if rel_drop(r_ptq8) > 0.10:
    print(f'PTQ lost {rel_drop(r_ptq8)*100:.1f} percent, running QAT with distillation')
    qat = build_ptq(fp32, 8); qat.train()
    for mm in qat.modules():
        if isinstance(mm, QAct): mm.calib = False
    o = torch.optim.Adam(qat.parameters(), lr=1e-4)
    Xt = torch.from_numpy(X[tr].astype(np.float32)); Yt = torch.from_numpy(to_unit(P[tr]).astype(np.float32))
    for ep in range(int(os.environ.get('KOLB_QAT_EPOCHS', '10'))):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), 128):
            b = perm[i:i+128]; xb = Xt[b].to(DEVICE); yb = Yt[b].to(DEVICE)
            with torch.no_grad(): tb = fp32(xb)
            out = qat(xb)
            loss = nn.functional.smooth_l1_loss(out, yb, beta=0.02) + nn.functional.smooth_l1_loss(out, tb, beta=0.02)
            o.zero_grad(); loss.backward(); o.step()
        qat.eval(); r = evaluate(qat, X[va].astype(np.float32), P[va]); qat.train(); print('QAT epoch', ep + 1, 'val', r)
    qat.eval(); r_qat = evaluate(qat, X[te].astype(np.float32), P[te]); print('QAT INT8 test MAE mm', r_qat)
    if np.mean(list(r_qat.values())) < np.mean(list(final_res.values())):
        final_name, final_q, final_res = 'QAT INT8 with distillation', qat, r_qat

In [5]:
# Optional stretch: INT4 weights with INT8 activations, PTQ only (just to show the trend)
q4 = build_ptq(fp32, wbits=4)
r4 = evaluate(q4, X[te].astype(np.float32), P[te])
print('INT4 weights / INT8 activations, PTQ only, test MAE mm:', r4)
n_w = sum(m.weight.numel() for m in fp32.modules() if isinstance(m, (nn.Conv2d, nn.Linear)))
res = {'fp32_test_mae_mm': ref, 'ptq_int8_test_mae_mm': r_ptq8, 'final_method': final_name, 'final_int8_test_mae_mm': final_res,
       'int4w_int8a_ptq_test_mae_mm': r4, 'weights': n_w, 'fp32_weight_kb': n_w * 4 / 1024, 'int8_weight_kb': n_w / 1024,
       'int4_weight_kb': n_w / 2 / 1024, 'fake_mode': FAKE_MODE,
       'note': 'Fake quantisation in PyTorch: accuracy effect only, not hardware speed. Sizes count conv and linear weights only.'}
torch.save(final_q.state_dict(), MODELS / 'oneshot_int8_fakequant.pt')
save_json(res, '06_quantisation.json'); print(json.dumps(res, indent=2))
log(f"06 quantisation: FP32 {ref} | {final_name} {final_res} | {n_w*4/1024:.0f} KB to {n_w/1024:.0f} KB")

INT4 weights / INT8 activations, PTQ only, test MAE mm:

 {'x': 3.354238510131836, 'y': 10.180859565734863, 'z': 5.1353583335876465, 'r': 2.5804314613342285}
saved C:\Users\Rey\Documents\Kolb_Radar_Project\outputs\results\06_quantisation.json
{
  "fp32_test_mae_mm": {
    "x": 2.7962286472320557,
    "y": 10.188100814819336,
    "z": 5.112585067749023,
    "r": 2.5817832946777344
  },
  "ptq_int8_test_mae_mm": {
    "x": 2.8913683891296387,
    "y": 10.187155723571777,
    "z": 5.141733169555664,
    "r": 2.584380626678467
  },
  "final_method": "PTQ INT8",
  "final_int8_test_mae_mm": {
    "x": 2.8913683891296387,
    "y": 10.187155723571777,
    "z": 5.141733169555664,
    "r": 2.584380626678467
  },
  "int4w_int8a_ptq_test_mae_mm": {
    "x": 3.354238510131836,
    "y": 10.180859565734863,
    "z": 5.1353583335876465,
    "r": 2.5804314613342285
  },
  "weights": 882496,
  "fp32_weight_kb": 3447.25,
  "int8_weight_kb": 861.8125,
  "int4_weight_kb": 430.90625,
  "fake_mode": false,
  "note": "Fake quantisation in PyTorch: accuracy effect o